In [4]:
from google.colab import files
import pandas as pd
import numpy as np
import hashlib
import os

uploaded = files.upload()

Saving peserta_pelatihan_umkm_2025.csv to peserta_pelatihan_umkm_2025.csv


In [5]:
FILE = 'peserta_pelatihan_umkm_2025.csv'

df = pd.read_csv(FILE)

print(df.shape)
df.head(3)

(620, 28)


,id_peserta,nama_lengkap,nik_dummy,no_hp,email,alamat_jalan,kelurahan,kecamatan,kota,tanggal_lahir,...,lama_usaha_tahun,jumlah_karyawan,omzet_usaha_bulanan,pernah_ikut_pelatihan,kanal_pendaftaran,tanggal_daftar,kehadiran_persen,skor_prates,skor_pascates,tingkat_kepuasan
0,PLT-2025-1309,Ratna Hidayat,99991405899249,8.000426e+10,ratna.hidayat@contoh.example,Jl. Cempaka No. 106 RT 02/RW 10,Sukatani,Tapos,Depok,1989-05-14,...,12.8,4,2950000,Tidak,Datang langsung,2025-04-04,75,52,67.0,5.0
1,PLT-2025-1512,Bunga Santoso,99992410874363,8.000347e+10,bunga.santoso@contoh.example,Jl. Bougenville No. 80 RT 10/RW 02,Mampang,Pancoran Mas,Depok,1987-10-24,...,6.5,3,7750000,Tidak,Formulir daring,2025-02-02,70,59,75.0,4.9
2,PLT-2025-1360,Endah Ramadhan,99991107987303,8.000266e+10,endah.ramadhan@contoh.example,Jl. Anggrek No. 138 RT 07/RW 08,Limo,Limo,Depok,1998-07-11,...,2.5,0,7550000,Tidak,WhatsApp,2025-03-26,90,51,61.0,4.9


In [10]:
# TODO A1

# Cetak nama kolom beserta tipe datanya
print(df.dtypes)

# Cetak jumlah baris dan kolom
print("Ukuran dataset:", df.shape)

id_peserta                      object
nama_lengkap                    object
nik_dummy                        int64
no_hp                          float64
email                           object
alamat_jalan                    object
kelurahan                       object
kecamatan                       object
kota                            object
tanggal_lahir                   object
jenis_kelamin                   object
pendidikan_terakhir             object
status_disabilitas              object
jumlah_anak                      int64
penghasilan_pribadi_bulanan      int64
nama_usaha                      object
kategori_usaha                  object
produk_utama                    object
lama_usaha_tahun               float64
jumlah_karyawan                  int64
omzet_usaha_bulanan              int64
pernah_ikut_pelatihan           object
kanal_pendaftaran               object
tanggal_daftar                  object
kehadiran_persen                 int64
skor_prates              

1. Perbaikan: Apakah peserta dengan kehadiran lebih tinggi memiliki skor pascates yang lebih tinggi?
   Kolom: kehadiran_persen, skor_pascates

2. Perbaikan: Apakah peserta dengan penghasilan pribadi bulanan lebih tinggi memiliki omzet usaha bulanan yang lebih tinggi?
   Kolom: penghasilan_pribadi_bulanan, omzet_usaha_bulanan

3. Mengapa uji kolom gagal: Dataset hanya memiliki kolom omzet_usaha_bulanan saat mendaftar. Tidak terdapat kolom omzet setelah pelatihan, sehingga pertanyaan apakah pelatihan menaikkan omzet tidak dapat dijawab.

In [12]:
# TODO A3

[c for c in df.columns if 'omzet' in c.lower()]

['omzet_usaha_bulanan']

B1 — Skala Pengukuran

Kolom	Skala	Alasan singkat
kategori_usaha	Nominal	Kategori usaha tidak memiliki urutan
pendidikan_terakhir	Ordinal	Jenjang pendidikan memiliki tingkatan
skor_prates	Interval	Perbedaan nilai skor dapat dibandingkan
omzet_usaha_bulanan	Rasio	Memiliki nilai nol dan dapat dibandingkan secara rasio
jumlah_karyawan	Rasio	Merupakan jumlah dengan nol absolut
tingkat_kepuasan	Ordinal	Tingkat kepuasan memiliki urutan
tanggal_daftar	Interval	Jarak antar tanggal bermakna
kecamatan	Nominal	Nama kecamatan tidak memiliki urutan

In [13]:
# TODO B2

urutan_pendidikan = [
    'SD',
    'SMP',
    'SMA/SMK',
    'D3',
    'S1',
    'S2'
]

df['pendidikan_terakhir'] = pd.Categorical(
    df['pendidikan_terakhir'],
    categories=urutan_pendidikan,
    ordered=True
)

print(df['pendidikan_terakhir'].value_counts().sort_index())

median_kode = df['pendidikan_terakhir'].cat.codes.median()

print("Median kode:", median_kode)
print("Median jenjang: SMA/SMK")

pendidikan_terakhir
SD          24
SMP         71
SMA/SMK    281
D3         107
S1         124
S2          13
Name: count, dtype: int64
Median kode: 2.0
Median jenjang: SMA/SMK


In [14]:
# TODO B3

df['tanggal_lahir'] = pd.to_datetime(df['tanggal_lahir'])
df['tanggal_daftar'] = pd.to_datetime(df['tanggal_daftar'])

print("Tanggal pendaftaran paling awal:",
      df['tanggal_daftar'].min())

print("Tanggal pendaftaran paling akhir:",
      df['tanggal_daftar'].max())

Tanggal pendaftaran paling awal: 2025-02-01 00:00:00
Tanggal pendaftaran paling akhir: 2025-05-31 00:00:00


In [15]:
# TODO C1 — bangkitkan kerangka kamus data

kamus = pd.DataFrame({
    'nama_kolom': df.columns,
    'tipe_data': df.dtypes.astype(str).values,
    'nilai_unik': [df[c].nunique(dropna=True) for c in df.columns],
    'nilai_kosong': [df[c].isna().sum() for c in df.columns],
    'contoh_nilai': [
        df[c].dropna().iloc[0] if df[c].notna().any() else np.nan
        for c in df.columns
    ]
})

kamus['deskripsi'] = ''
kamus['skala_pengukuran'] = ''
kamus['kategori_data_pribadi'] = ''
kamus['tindakan_penanganan'] = ''

print(kamus.shape)
kamus

(28, 9)


,nama_kolom,tipe_data,nilai_unik,nilai_kosong,contoh_nilai,deskripsi,skala_pengukuran,kategori_data_pribadi,tindakan_penanganan
0,id_peserta,object,620,0,PLT-2025-1309,,,,
1,nama_lengkap,object,450,0,Ratna Hidayat,,,,
2,nik_dummy,int64,620,0,99991405899249,,,,
3,no_hp,float64,611,9,80004260813.0,,,,
4,email,object,450,0,ratna.hidayat@contoh.example,,,,
5,alamat_jalan,object,620,0,Jl. Cempaka No. 106 RT 02/RW 10,,,,
6,kelurahan,object,26,0,Sukatani,,,,
7,kecamatan,object,19,0,Tapos,,,,
8,kota,object,1,0,Depok,,,,
9,tanggal_lahir,datetime64[ns],598,0,1989-05-14 00:00:00,,,,


In [16]:
# TODO C2 — kolom mana yang punya nilai kosong?

kamus[kamus['nilai_kosong'] > 0]

,nama_kolom,tipe_data,nilai_unik,nilai_kosong,contoh_nilai,deskripsi,skala_pengukuran,kategori_data_pribadi,tindakan_penanganan
3,no_hp,float64,611,9,80004260813.0,,,,
26,skor_pascates,float64,64,34,67.0,,,,
27,tingkat_kepuasan,float64,28,18,5.0,,,,


C3 — Mengisi Kamus Data


TODO C3 — isi kamus untuk lima kolom

Isi tabelnya:

Kolom	Deskripsi	Skala	Satuan
lama_usaha_tahun	Menunjukkan lamanya usaha peserta telah berjalan	Rasio	Tahun
kehadiran_persen	Menunjukkan persentase kehadiran peserta selama pelatihan	Rasio	Persen (%)
skor_pascates	Menunjukkan skor yang diperoleh peserta setelah mengikuti pelatihan	Interval	Skor
kanal_pendaftaran	Menunjukkan kanal yang digunakan peserta untuk melakukan pendaftaran	Nominal	-
jumlah_anak	Menunjukkan jumlah anak yang dimiliki peserta	Rasio	Orang

In [17]:
# TODO D1 — susun tabel penandaan Anda sendiri

PENANDAAN = {
    'id_peserta': ('pengenal langsung', 'hash'),
    'nama_lengkap': ('pengenal langsung', 'hapus'),
    'nik_dummy': ('pengenal langsung', 'hapus'),
    'no_hp': ('pengenal langsung', 'hapus'),
    'email': ('pengenal langsung', 'hapus'),
    'alamat_jalan': ('pengenal langsung', 'hapus'),

    'kelurahan': ('quasi-identifier', 'generalisasi'),
    'kecamatan': ('quasi-identifier', 'generalisasi'),
    'tanggal_lahir': ('quasi-identifier', 'generalisasi'),
    'jenis_kelamin': ('quasi-identifier', 'pertahankan'),
    'pendidikan_terakhir': ('quasi-identifier', 'pertahankan'),

    'status_disabilitas': ('data spesifik', 'hapus'),
    'penghasilan_pribadi_bulanan': ('data spesifik', 'generalisasi'),
    'jumlah_anak': ('data spesifik', 'generalisasi'),

    'nama_usaha': ('quasi-identifier', 'generalisasi')
}

tanda = pd.DataFrame([
    {
        'nama_kolom': c,
        'kategori': PENANDAAN.get(
            c,
            ('bukan data pribadi', 'pertahankan')
        )[0],
        'tindakan': PENANDAAN.get(
            c,
            ('bukan data pribadi', 'pertahankan')
        )[1]
    }
    for c in df.columns
])

print(tanda)

print("\nJumlah kolom per kategori:")
print(tanda['kategori'].value_counts())

                     nama_kolom            kategori      tindakan
0                    id_peserta   pengenal langsung          hash
1                  nama_lengkap   pengenal langsung         hapus
2                     nik_dummy   pengenal langsung         hapus
3                         no_hp   pengenal langsung         hapus
4                         email   pengenal langsung         hapus
5                  alamat_jalan   pengenal langsung         hapus
6                     kelurahan    quasi-identifier  generalisasi
7                     kecamatan    quasi-identifier  generalisasi
8                          kota  bukan data pribadi   pertahankan
9                 tanggal_lahir    quasi-identifier  generalisasi
10                jenis_kelamin    quasi-identifier   pertahankan
11          pendidikan_terakhir    quasi-identifier   pertahankan
12           status_disabilitas       data spesifik         hapus
13                  jumlah_anak       data spesifik  generalisasi
14  pengha

D2 — Jawaban Markdown



Kolom	Kategori pilihan Anda	Tindakan	Alasan
nama_usaha	quasi-identifier	generalisasi	Nama usaha dapat dicari melalui mesin pencari dan dapat digabungkan dengan informasi lain untuk mengarah kepada pemilik usaha.
omzet_usaha_bulanan	bukan data pribadi	pertahankan	Data ini menggambarkan kondisi usaha dan digunakan sebagai bahan analisis usaha, bukan sebagai pengenal langsung seseorang.
id_peserta	pengenal langsung	hash	ID digunakan untuk membedakan satu peserta dengan peserta lain, tetapi nilai aslinya tidak perlu dipublikasikan.

Perhatikan bahwa alasan nama_usaha, omzet_usaha_bulanan, dan id_peserta memang diminta untuk dipertanggungjawabkan oleh soal, bukan sekadar diberi label.

In [19]:
# TODO E1 — HAPUS

# Kolom yang memiliki tindakan "hapus"
kolom_hapus = [
    c for c in df.columns
    if PENANDAAN.get(c, ('bukan data pribadi', 'pertahankan'))[1] == 'hapus'
]

# Buat data baru tanpa kolom tersebut
aman = df.drop(columns=kolom_hapus).copy()

print("Kolom yang dibuang:", len(kolom_hapus))
print("Kolom yang tersisa:", len(aman.columns))
print("Kolom yang dibuang:", kolom_hapus)

Kolom yang dibuang: 6
Kolom yang tersisa: 22
Kolom yang dibuang: ['nama_lengkap', 'nik_dummy', 'no_hp', 'email', 'alamat_jalan', 'status_disabilitas']


In [20]:
# TODO E2 — HASH

def samarkan(nilai, garam):
    if pd.isna(nilai):
        return np.nan

    teks = str(nilai)

    hasil = hashlib.sha256(
        (garam + teks).encode('utf-8')
    ).hexdigest()

    return hasil[:12]


GARAM = 'ganti-dengan-kalimat-rahasia-tim'

# Cari kolom yang tindakannya adalah hash
kolom_hash = [
    c for c in df.columns
    if PENANDAAN.get(c, ('bukan data pribadi', 'pertahankan'))[1] == 'hash'
]

# Terapkan hash
for c in kolom_hash:
    aman[c] = aman[c].apply(
        lambda x: samarkan(x, GARAM)
    )

print("Kolom yang di-hash:", kolom_hash)

# Tampilkan beberapa hasil
print(aman[kolom_hash].head())

Kolom yang di-hash: ['id_peserta']
     id_peserta
0  09b72e9a47af
1  565439020ae0
2  2398346a6926
3  97d7aae2281c
4  5695b53f5a0a


In [21]:
# TODO E3 — buktikan hash bersifat konsisten dan tidak terbaca

nilai1 = df['id_peserta'].iloc[0]
nilai2 = df['id_peserta'].iloc[1]

hash1 = samarkan(nilai1, GARAM)
hash1_lagi = samarkan(nilai1, GARAM)
hash2 = samarkan(nilai2, GARAM)

print("Hash nilai pertama :", hash1)
print("Hash nilai pertama lagi:", hash1_lagi)
print("Hash nilai kedua   :", hash2)

print("\nNilai yang sama menghasilkan hash sama:",
      hash1 == hash1_lagi)

print("Nilai yang berbeda menghasilkan hash berbeda:",
      hash1 != hash2)

Hash nilai pertama : 09b72e9a47af
Hash nilai pertama lagi: 09b72e9a47af
Hash nilai kedua   : 565439020ae0

Nilai yang sama menghasilkan hash sama: True
Nilai yang berbeda menghasilkan hash berbeda: True


Jawaban E3 — Markdown

kalau GARAM ikut ter-commit ke GitHub, apa yang masih terlindungi dan apa yang tidak?

Jika GARAM ikut ter-commit ke GitHub, nilai asli masih tidak langsung terlihat karena yang disimpan adalah hasil hash. Namun perlindungannya menjadi lebih lemah karena orang lain yang mengetahui nilai kandidat dapat menghitung ulang hash menggunakan GARAM tersebut dan mencocokkannya dengan data yang dipublikasikan.

In [22]:
# TODO E4 — GENERALISASI

ACUAN = pd.Timestamp('2025-09-01')

# Pastikan tanggal lahir bertipe datetime
aman['tanggal_lahir'] = pd.to_datetime(aman['tanggal_lahir'])

# Hitung usia pada tanggal 1 September 2025
aman['usia'] = (
    ACUAN.year - aman['tanggal_lahir'].dt.year
    - (
        (ACUAN.month < aman['tanggal_lahir'].dt.month) |
        (
            (ACUAN.month == aman['tanggal_lahir'].dt.month) &
            (ACUAN.day < aman['tanggal_lahir'].dt.day)
        )
    ).astype(int)
)

# Membuat kelompok umur
bins = [-np.inf, 25, 35, 45, 55, np.inf]

labels = [
    '<25',
    '25-34',
    '35-44',
    '45-54',
    '55+'
]

aman['kelompok_umur'] = pd.cut(
    aman['usia'],
    bins=bins,
    labels=labels,
    right=False
)

# Buang tanggal lahir dan kelurahan
aman = aman.drop(
    columns=['tanggal_lahir', 'kelurahan']
)

# Rapikan penulisan kecamatan
aman['kecamatan'] = aman['kecamatan'].str.title()

print("Jumlah setiap kelompok umur:")
print(aman['kelompok_umur'].value_counts().sort_index())

print("\nJumlah kecamatan:")
print(aman['kecamatan'].nunique())

print("\nDaftar kecamatan:")
print(sorted(aman['kecamatan'].unique()))

Jumlah setiap kelompok umur:
kelompok_umur
<25       53
25-34    153
35-44    267
45-54    108
55+       39
Name: count, dtype: int64

Jumlah kecamatan:
10

Daftar kecamatan:
['Beji', 'Bojongsari', 'Cimanggis', 'Cinere', 'Cipayung', 'Limo', 'Pancoran Mas', 'Sawangan', 'Sukmajaya', 'Tapos']


In [23]:
# TODO F1 — tulis fungsi cek_k(data, kolom)

def cek_k(data, kolom):
    # Hitung jumlah orang pada setiap kombinasi kolom
    grup = data.groupby(
        kolom,
        dropna=False,
        observed=True
    ).size()

    # Nilai k = jumlah terkecil dalam sebuah grup
    k = grup.min()

    # Jumlah grup yang hanya berisi 1 orang
    grup_satu = (grup == 1).sum()

    # Jumlah seluruh grup
    total_grup = len(grup)

    return pd.Series({
        'k': k,
        'grup berisi 1 orang': grup_satu,
        'total grup': total_grup
    })

In [24]:
# TODO F2 — ukur risiko SEBELUM penanganan

kolom_qi_awal = [
    'kelurahan',
    'jenis_kelamin',
    'pendidikan_terakhir',
    'tanggal_lahir'
]

hasil_f2 = cek_k(df, kolom_qi_awal)

print(hasil_f2)

k                        1
grup berisi 1 orang    620
total grup             620
dtype: int64


Jawaban Markdown F2

Walaupun keempat kolom tersebut tidak memuat nama maupun nomor telepon, kombinasi kelurahan, jenis kelamin, pendidikan, dan tanggal lahir dapat menjadi sangat unik. Jika informasi tersebut digabungkan dengan informasi dari sumber lain, peserta dapat berpotensi diidentifikasi kembali.

F3 — Risiko SETELAH generalisasi

In [31]:
kolom_qi_setelah = [
    'kecamatan',
    'jenis_kelamin',
    'pendidikan_terakhir',
    'kelompok_umur'
]

hasil_f3 = cek_k(aman, kolom_qi_setelah)

print(hasil_f3)

k                        1
grup berisi 1 orang    138
total grup             266
dtype: int64


F4 — Naikkan k sampai ≥ 5

In [32]:
kolom_qi_final = [
    'kecamatan',
    'jenis_kelamin'
]

hasil_f4 = cek_k(aman, kolom_qi_final)

print(hasil_f4)


k                       9
grup berisi 1 orang     0
total grup             20
dtype: int64


In [27]:
# TODO F5 — sel kecil pada tabel agregat

tabel_silang = pd.crosstab(
    aman['kecamatan'],
    aman['kategori_usaha']
)

print(tabel_silang)

# Cari sel yang terisi tetapi jumlahnya kurang dari 5
sel_kecil = tabel_silang[
    (tabel_silang > 0) & (tabel_silang < 5)
]

jumlah_sel_kecil = sel_kecil.count().sum()

print("\nJumlah sel terisi dengan peserta < 5:",
      jumlah_sel_kecil)

kategori_usaha  Fesyen  Jasa Digital  Kriya  Kuliner  Pertanian
kecamatan                                                      
Beji                14            10     15       42          6
Bojongsari           7             5      4       23          2
Cimanggis           16             8     15       44          7
Cinere              16            11      8       23          9
Cipayung             4             6      5       10          1
Limo                10             4     10       15          4
Pancoran Mas        25            12     19       31         12
Sawangan             5             3      3       10          2
Sukmajaya           22            14     11       36         12
Tapos               11             5     11       16          6

Jumlah sel terisi dengan peserta < 5: 9


In [33]:
from google.colab import files

uploaded = files.upload()

Saving Data Stunting Indonesia (1).csv to Data Stunting Indonesia (1).csv


In [34]:
import pandas as pd

FILE_PROYEK = 'Data Stunting Indonesia (1).csv'

data_proyek = pd.read_csv(FILE_PROYEK)

print("Ukuran dataset:", data_proyek.shape)
print("\nNama kolom:")
print(data_proyek.columns.tolist())

print("\n5 data pertama:")
display(data_proyek.head())

Ukuran dataset: (38, 1)

Nama kolom:
['Provinsi;2020;2021;2022;2023;']

5 data pertama:


,,,,Provinsi;2020;2021;2022;2023;
ACEH;13,4;12,1;8,0;7,5;
SUMATERA UTARA;6,8;6,7;5,5;4,5;
SUMATERA BARAT;17,5;15,1;10,3;8,8;
RIAU;7,7;6,0;4,2;3,0;
JAMBI;7,6;3,0;4,1;3,7;


In [35]:
data_proyek = pd.read_csv(
    FILE_PROYEK,
    sep=';'
)

print("Ukuran dataset:", data_proyek.shape)
print("\nNama kolom:")
print(data_proyek.columns.tolist())

display(data_proyek.head())

Ukuran dataset: (38, 6)

Nama kolom:
['Provinsi', '2020', '2021', '2022', '2023', 'Unnamed: 5']


,Provinsi,2020,2021,2022,2023,Unnamed: 5
0,ACEH,"13,4","12,1","8,0","7,5",NaN
1,SUMATERA UTARA,"6,8","6,7","5,5","4,5",NaN
2,SUMATERA BARAT,"17,5","15,1","10,3","8,8",NaN
3,RIAU,"7,7","6,0","4,2","3,0",NaN
4,JAMBI,"7,6","3,0","4,1","3,7",NaN


In [36]:
data_proyek = data_proyek.drop(columns=['Unnamed: 5'])

print("Ukuran dataset:", data_proyek.shape)
print("\nNama kolom:")
print(data_proyek.columns.tolist())

display(data_proyek.head())

Ukuran dataset: (38, 5)

Nama kolom:
['Provinsi', '2020', '2021', '2022', '2023']


,Provinsi,2020,2021,2022,2023
0,ACEH,"13,4","12,1","8,0","7,5"
1,SUMATERA UTARA,"6,8","6,7","5,5","4,5"
2,SUMATERA BARAT,"17,5","15,1","10,3","8,8"
3,RIAU,"7,7","6,0","4,2","3,0"
4,JAMBI,"7,6","3,0","4,1","3,7"


In [37]:
print(data_proyek.dtypes)

Provinsi    object
2020        object
2021        object
2022        object
2023        object
dtype: object


In [38]:
# Ubah kolom tahun menjadi angka
kolom_tahun = ['2020', '2021', '2022', '2023']

for kolom in kolom_tahun:
    data_proyek[kolom] = (
        data_proyek[kolom]
        .astype(str)
        .str.replace(',', '.', regex=False)
        .astype(float)
    )

print(data_proyek.dtypes)

Provinsi     object
2020        float64
2021        float64
2022        float64
2023        float64
dtype: object


In [39]:
display(data_proyek.head())

,Provinsi,2020,2021,2022,2023
0,ACEH,13.4,12.1,8.0,7.5
1,SUMATERA UTARA,6.8,6.7,5.5,4.5
2,SUMATERA BARAT,17.5,15.1,10.3,8.8
3,RIAU,7.7,6.0,4.2,3.0
4,JAMBI,7.6,3.0,4.1,3.7


### G1 — Pertanyaan Analisis

1. Bagaimana perubahan prevalensi stunting di setiap provinsi dari tahun 2020 sampai 2023?

2. Provinsi mana yang memiliki prevalensi stunting tertinggi pada tahun 2023?

3. Provinsi mana yang mengalami penurunan prevalensi stunting terbesar dari tahun 2020 sampai 2023?

4. Bagaimana rata-rata prevalensi stunting seluruh provinsi pada periode 2020–2023?



### G2 — Provenance Dataset

| Informasi           | Isi                                                          |
| ------------------- | ------------------------------------------------------------ |
| Nama sumber         | Kaggle                                                       |
| Link sumber         | Link halaman dataset Kaggle yang digunakan                   |
| Tanggal pengambilan | Tanggal kamu mengunduh dataset                               |
| Lisensi/izin        | Mengikuti lisensi yang tercantum pada halaman dataset Kaggle |
| Catatan             | Dataset                                                      |


G3 adalah Kamus Data untuk dataset proyek.

In [46]:
# G3 — Membuat Kamus Data Proyek

# Deskripsi manual untuk masing-masing kolom
deskripsi_kolom = {
    'Provinsi': 'Nama provinsi di Indonesia',
    '2020': 'Prevalensi stunting (%) pada tahun 2020',
    '2021': 'Prevalensi stunting (%) pada tahun 2021',
    '2022': 'Prevalensi stunting (%) pada tahun 2022',
    '2023': 'Prevalensi stunting (%) pada tahun 2023'
}

skala_kolom = {
    'Provinsi': 'Nominal',
    '2020': 'Rasio',
    '2021': 'Rasio',
    '2022': 'Rasio',
    '2023': 'Rasio'
}

kamus_proyek = pd.DataFrame({
    'nama_kolom': list(data_proyek.columns),
    'tipe_data': [str(data_proyek[c].dtype) for c in data_proyek.columns],
    'nilai_unik': [data_proyek[c].nunique(dropna=True) for c in data_proyek.columns],
    'nilai_kosong': [data_proyek[c].isna().sum() for c in data_proyek.columns],
    'contoh_nilai': [
        data_proyek[c].dropna().iloc[0]
        if data_proyek[c].notna().any()
        else np.nan
        for c in data_proyek.columns
    ],
    'deskripsi': [deskripsi_kolom.get(c, '') for c in data_proyek.columns],
    'skala_pengukuran': [skala_kolom.get(c, '') for c in data_proyek.columns],
    'kategori_data_pribadi': ['bukan data pribadi'] * len(data_proyek.columns),
    'tindakan_penanganan': ['pertahankan'] * len(data_proyek.columns)
})

display(kamus_proyek)

,nama_kolom,tipe_data,nilai_unik,nilai_kosong,contoh_nilai,deskripsi,skala_pengukuran,kategori_data_pribadi,tindakan_penanganan
0,Provinsi,object,38,0,ACEH,Nama provinsi di Indonesia,Nominal,bukan data pribadi,pertahankan
1,2020,float64,34,0,13.4,Prevalensi stunting (%) pada tahun 2020,Rasio,bukan data pribadi,pertahankan
2,2021,float64,34,0,12.1,Prevalensi stunting (%) pada tahun 2021,Rasio,bukan data pribadi,pertahankan
3,2022,float64,36,0,8.0,Prevalensi stunting (%) pada tahun 2022,Rasio,bukan data pribadi,pertahankan
4,2023,float64,33,0,7.5,Prevalensi stunting (%) pada tahun 2023,Rasio,bukan data pribadi,pertahankan


In [47]:
# G4 — Pemeriksaan data proyek

print("Ukuran data:", data_proyek.shape)

print("\nJumlah data kosong:")
print(data_proyek.isna().sum())

print("\nData duplikat:", data_proyek.duplicated().sum())

print("\nNama provinsi duplikat:")
print(data_proyek['Provinsi'].duplicated().sum())

print("\nDaftar provinsi:")
print(data_proyek['Provinsi'].tolist())

Ukuran data: (38, 5)

Jumlah data kosong:
Provinsi    0
2020        0
2021        0
2022        0
2023        0
dtype: int64

Data duplikat: 0

Nama provinsi duplikat:
0

Daftar provinsi:
['ACEH', 'SUMATERA UTARA', 'SUMATERA BARAT', 'RIAU', 'JAMBI', 'SUMATERA SELATAN', 'BENGKULU', 'LAMPUNG', 'KEPULAUAN BANGKA BELITUNG', 'KEPULAUAN RIAU', 'DKI JAKARTA', 'JAWA BARAT', 'JAWA TENGAH', 'DI YOGYAKARTA', 'JAWA TIMUR', 'BANTEN', 'BALI', 'NUSA TENGGARA BARAT', 'NUSA TENGGARA TIMUR', 'KALIMANTAN BARAT', 'KALIMANTAN TENGAH', 'KALIMANTAN SELATAN', 'KALIMANTAN TIMUR', 'KALIMANTAN UTARA', 'SULAWESI UTARA', 'SULAWESI TENGAH', 'SULAWESI SELATAN', 'SULAWESI TENGGARA', 'GORONTALO', 'SULAWESI BARAT', 'MALUKU', 'MALUKU UTARA', 'PAPUA BARAT', 'PAPUA', 'PAPUA PEGUNUNGAN', 'PAPUA SELATAN', 'PAPUA TENGAH', 'PAPUA BARAT DAYA']


In [48]:
# Cek rentang nilai prevalensi stunting

for tahun in ['2020', '2021', '2022', '2023']:
    print(
        tahun,
        "min =", data_proyek[tahun].min(),
        "| max =", data_proyek[tahun].max()
    )

2020 min = 0.1 | max = 28.7
2021 min = 3.0 | max = 22.6
2022 min = 1.2 | max = 23.1
2023 min = 1.0 | max = 23.2


### G5 — Checklist Sebelum Git Push

- [x] Tidak ada nama lengkap peserta
- [x] Tidak ada NIK/nomor identitas
- [x] Tidak ada nomor telepon
- [x] Tidak ada alamat rumah
- [x] Tidak ada alamat email
- [x] Tidak ada data individu yang dapat digunakan untuk identifikasi langsung
- [x] Kolom data sudah didokumentasikan dalam kamus data
- [x] Tipe data sudah diperiksa
- [x] Nilai kosong sudah diperiksa
- [x] Data duplikat sudah diperiksa
- [x] Data numerik sudah dikonversi menjadi tipe numerik
- [x] Dataset menggunakan tingkat agregasi provinsi
- [x] Tidak ada secret/GARAM yang dimasukkan ke dataset
- [x] Tidak ada file atau informasi sensitif yang perlu dipublikasikan